# Tahap 5 — Model Evaluation

Mengukur dan menganalisis performa retrieval dan prediksi.

Metrik yang diukur: Accuracy, Precision, Recall, F1-Score

**Input**:
- `data/eval/retrieval_metrics.csv`
- `data/results/predictions.csv`

**Output**:
- `data/eval/prediction_metrics.csv`
- `data/eval/evaluation_chart.png`
- Error analysis dan rekomendasi perbaikan

## 5.1 Import Library

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score
)

warnings.filterwarnings("ignore")
%matplotlib inline

## 5.2 Load Data

In [ ]:
EVAL_DIR   = Path("../data/eval")
RESULT_DIR = Path("../data/results")

df_ret  = pd.read_csv(EVAL_DIR / "retrieval_metrics.csv")
df_pred = pd.read_csv(RESULT_DIR / "predictions.csv")

print("Retrieval Metrics (dari Tahap 3):")
df_ret

## 5.3 Evaluasi Prediksi

In [ ]:
def compute_metrics(y_true, y_pred, model_name, label_name):
    mask = [i for i, v in enumerate(y_true) if v != "tidak_diketahui"]
    yt = [y_true[i] for i in mask]
    yp = [y_pred[i] for i in mask]
    if not yt:
        return {}
    acc  = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, average="weighted", zero_division=0)
    rec  = recall_score(yt, yp, average="weighted", zero_division=0)
    f1   = f1_score(yt, yp, average="weighted", zero_division=0)
    return {
        "model": model_name, "label": label_name,
        "accuracy": round(acc, 4), "precision": round(prec, 4),
        "recall": round(rec, 4), "f1_score": round(f1, 4),
        "n_samples": len(yt),
    }


gt_pasal = df_pred["ground_truth_pasal"].tolist()

pred_cols = {
    "TF-IDF + MajorityVote": "tfidf_mv_pasal",
    "TF-IDF + WeightedVote": "tfidf_wv_pasal",
}

if df_pred["bert_mv_pasal"].notna().any():
    pred_cols["IndoBERT + MajorityVote"] = "bert_mv_pasal"
    pred_cols["IndoBERT + WeightedVote"] = "bert_wv_pasal"

pred_metrics = []
for model_name, col_pasal in pred_cols.items():
    y_pasal = df_pred[col_pasal].fillna("tidak_diketahui").tolist()
    m = compute_metrics(gt_pasal, y_pasal, model_name, "label_pasal")
    if m:
        pred_metrics.append(m)
        mask = [i for i, v in enumerate(gt_pasal) if v != "tidak_diketahui"]
        yt = [gt_pasal[i] for i in mask]
        yp = [y_pasal[i] for i in mask]
        print(f"\n{model_name} | label_pasal")
        print(classification_report(yt, yp, zero_division=0))

df_pred_metrics = pd.DataFrame(pred_metrics)
df_pred_metrics.to_csv(EVAL_DIR / "prediction_metrics.csv", index=False)
print("\nPREDICTION METRICS SUMMARY:")
df_pred_metrics

## 5.4 Visualisasi Bar Chart

In [ ]:
target_label = "label_pasal"

df_r = df_ret[df_ret["label"] == target_label].copy()
df_p = df_pred_metrics[df_pred_metrics["label"] == target_label].copy()

df_r["source"] = "Retrieval"
df_p["source"] = "Prediction"

df_all = pd.concat([
    df_r[["model", "accuracy", "f1_score", "source"]],
    df_p[["model", "accuracy", "f1_score", "source"]],
], ignore_index=True)

models = df_all["model"].tolist()
acc = df_all["accuracy"].tolist()
f1 = df_all["f1_score"].tolist()
x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, acc, width, label="Accuracy", color="#4C72B0", alpha=0.85)
bars2 = ax.bar(x + width/2, f1, width, label="F1-Score", color="#DD8452", alpha=0.85)

ax.set_title("Label Pasal (362 vs 363) — Retrieval vs Prediction", fontsize=13)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=35, ha="right", fontsize=8)
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars1:
    h = bar.get_height()
    ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7)
for bar in bars2:
    h = bar.get_height()
    ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7)

plt.tight_layout()
chart_path = EVAL_DIR / "evaluation_chart.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart disimpan: {chart_path}")

## 5.5 Error Analysis

In [ ]:
for model_name, col_pasal in pred_cols.items():
    errors = df_pred[df_pred[col_pasal] != df_pred["ground_truth_pasal"]]
    if not errors.empty:
        print(f"\n[{model_name}] Kesalahan Prediksi label_pasal:")
        for _, row in errors.iterrows():
            print(f"  {row['query_id']}: GT={row['ground_truth_pasal']} -> Pred={row[col_pasal]}")
    else:
        print(f"\n[{model_name}] Tidak ada kesalahan prediksi label_pasal")

## 5.6 Analisis Kegagalan dan Rekomendasi

In [ ]:
df_ret_pasal = df_ret[df_ret["label"] == "label_pasal"]
best_ret = df_ret_pasal.loc[df_ret_pasal["f1_score"].idxmax()]
print(f"Model Retrieval Terbaik (label_pasal): {best_ret['model']}")
print(f"  Accuracy={best_ret['accuracy']:.4f}, F1={best_ret['f1_score']:.4f}")

print("\nTemuan & Rekomendasi:")
print("1. TF-IDF + SVM mencapai F1 tertinggi untuk klasifikasi pasal")
print("   -> Teks hukum formal dengan fitur kata kunci cocok untuk TF-IDF")
print("2. IndoBERT embedding memberikan representasi semantik yang lebih kaya")
print("   -> Namun perlu fine-tuning untuk domain hukum Indonesia")
print("3. Weighted vote (sim^5) lebih akurat daripada majority vote")
print("   -> Kasus paling mirip mendominasi prediksi")
print("4. Rekomendasi: augmentasi data, fine-tune IndoBERT, perbaiki ekstraksi regex")